In [35]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, classification_report

In [36]:
df = pd.read_csv(r"C:\Projetos\advanced-classification-ml\data\raw\WA_Fn-UseC_-Telco-Customer-Churn.csv")

In [37]:
df.shape

(7043, 21)

In [38]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   str    
 1   gender            7043 non-null   str    
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   str    
 4   Dependents        7043 non-null   str    
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   str    
 7   MultipleLines     7043 non-null   str    
 8   InternetService   7043 non-null   str    
 9   OnlineSecurity    7043 non-null   str    
 10  OnlineBackup      7043 non-null   str    
 11  DeviceProtection  7043 non-null   str    
 12  TechSupport       7043 non-null   str    
 13  StreamingTV       7043 non-null   str    
 14  StreamingMovies   7043 non-null   str    
 15  Contract          7043 non-null   str    
 16  PaperlessBilling  7043 non-null   str    
 17  Paymen

In [39]:
df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [40]:
df["Churn"].value_counts(normalize=True)

Churn
No     0.73463
Yes    0.26537
Name: proportion, dtype: float64

In [41]:
df["Churn"] = df["Churn"].map({"Yes": 1, "No": 0})

In [42]:
df = df.drop(columns=["customerID"])

In [43]:
X = df.drop("Churn", axis=1)
y = df["Churn"]

In [44]:
X = pd.get_dummies(X)

In [45]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [46]:
X_train.shape, X_test.shape

((5634, 6575), (1409, 6575))

In [47]:
from sklearn.linear_model import LogisticRegression

log_model = LogisticRegression(max_iter=1000)
log_model.fit(X_train, y_train)

y_pred_log = log_model.predict(X_test)

In [48]:
from sklearn.metrics import accuracy_score, classification_report

print("Acurácia - Regressão Logística:", accuracy_score(y_test, y_pred_log))
print(classification_report(y_test, y_pred_log))

Acurácia - Regressão Logística: 0.7934705464868701
              precision    recall  f1-score   support

           0       0.84      0.89      0.86      1035
           1       0.63      0.54      0.58       374

    accuracy                           0.79      1409
   macro avg       0.74      0.71      0.72      1409
weighted avg       0.79      0.79      0.79      1409



In [49]:
from sklearn.tree import DecisionTreeClassifier

tree_model = DecisionTreeClassifier(random_state=42)
tree_model.fit(X_train, y_train)

y_pred_tree = tree_model.predict(X_test)


In [50]:
print("Acurácia - Árvore de Decisão:", accuracy_score(y_test, y_pred_tree))
print(classification_report(y_test, y_pred_tree))

Acurácia - Árvore de Decisão: 0.7700496806245565
              precision    recall  f1-score   support

           0       0.83      0.87      0.85      1035
           1       0.58      0.50      0.53       374

    accuracy                           0.77      1409
   macro avg       0.70      0.68      0.69      1409
weighted avg       0.76      0.77      0.76      1409



In [51]:
from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(random_state=42)
rf_model.fit(X_train, y_train)

y_pred_rf = rf_model.predict(X_test)

In [52]:
print("Acurácia - Logistic Regression:", accuracy_score(y_test, y_pred_log))
print("Acurácia - Decision Tree:", accuracy_score(y_test, y_pred_tree))
print("Acurácia - Random Forest:", accuracy_score(y_test, y_pred_rf))

Acurácia - Logistic Regression: 0.7934705464868701
Acurácia - Decision Tree: 0.7700496806245565
Acurácia - Random Forest: 0.7828246983676366


# 📊 Avaliação dos Modelos (Baseline)

Nesta etapa, realizamos a comparação de desempenho entre diferentes algoritmos de classificação aplicados ao problema de churn.

O objetivo é estabelecer uma linha de base de performance utilizando modelos tradicionais antes da evolução para abordagens mais avançadas com XGBoost e pipelines.

In [53]:
from xgboost import XGBClassifier
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

In [54]:
X = df.drop("Churn", axis=1)
y = df["Churn"]

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [55]:
categorical_features = X.select_dtypes(include=["object"]).columns

preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features)
    ],
    remainder="passthrough"
)

C:\Users\vagne\AppData\Local\Temp\ipykernel_9576\724742871.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_features = X.select_dtypes(include=["object"]).columns


In [56]:
xgb_model = XGBClassifier(
    n_estimators=200,
    learning_rate=0.1,
    max_depth=4,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    eval_metric="logloss"
)

model_xgb = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", xgb_model)
])

In [57]:
model_xgb.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessor', ...), ('classifier', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[int64](2,)","[0,1]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](19,)","['gender','SeniorCitizen','Partner',...,'PaymentMethod','MonthlyCharges', 'TotalCharges']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,19
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough

In [58]:
y_pred_xgb = model_xgb.predict(X_test)

In [59]:
print("Acurácia - Logistic Regression:", accuracy_score(y_test, y_pred_log))
print("Acurácia - Decision Tree:", accuracy_score(y_test, y_pred_tree))
print("Acurácia - Random Forest:", accuracy_score(y_test, y_pred_rf))
print("Acurácia - XGBoost (Pipeline):", accuracy_score(y_test, y_pred_xgb))

Acurácia - Logistic Regression: 0.7934705464868701
Acurácia - Decision Tree: 0.7700496806245565
Acurácia - Random Forest: 0.7828246983676366
Acurácia - XGBoost (Pipeline): 0.7991483321504613


# 📊 Resultados com XGBoost em Pipeline

Após a implementação do XGBoost integrado a um pipeline de pré-processamento, foi possível observar uma leve melhoria de performance em relação aos modelos tradicionais testados anteriormente.

O modelo baseado em Gradient Boosting apresentou o melhor desempenho geral, reforçando a eficácia de técnicas de ensemble learning em problemas de classificação.

Essa etapa também consolidou a importância do uso de pipelines para garantir consistência no fluxo de Machine Learning.

# ==========================================================
# Aprendizado Semi-Supervisionado
# ==========================================================

Nesta etapa do projeto, serão exploradas técnicas de aprendizado semi-supervisionado, cenário comum em aplicações reais onde apenas uma parte dos dados possui rótulos conhecidos.

Serão aplicadas duas abordagens principais:

- Self Training (Pseudo Labeling)
- Label Propagation

Ao final, os resultados serão comparados com os modelos supervisionados desenvolvidos nas etapas anteriores, permitindo avaliar o impacto da utilização de dados não rotulados na performance do modelo.

## Objetivo desta etapa

Em muitos problemas reais de Ciência de Dados, obter dados é relativamente simples, porém rotulá-los manualmente pode ser caro, demorado ou inviável.

O aprendizado semi-supervisionado busca aproveitar uma pequena quantidade de dados rotulados juntamente com um grande volume de dados sem rótulo para melhorar o desempenho dos modelos de Machine Learning.

Neste projeto será simulado esse cenário utilizando o conjunto de treinamento já preparado nas etapas anteriores.

In [60]:
import numpy as np

from sklearn.semi_supervised import SelfTrainingClassifier
from sklearn.semi_supervised import LabelPropagation

from sklearn.metrics import accuracy_score
from sklearn.metrics import classification_report

## Simulação de dados não rotulados

Como o dataset Telco Customer Churn já possui todos os rótulos conhecidos, será necessário simular um cenário semi-supervisionado.

Para isso, uma parte dos rótulos do conjunto de treinamento será removida temporariamente, sendo representada pelo valor **-1**, conforme esperado pelos algoritmos semi-supervisionados da biblioteca Scikit-Learn.

In [61]:
# Mantém a reprodutibilidade
np.random.seed(42)

# Copiando os rótulos do treinamento
y_train_semi = y_train.copy()

# Percentual de amostras que permanecerão sem rótulo
unlabeled_percentage = 0.70

# Índices escolhidos aleatoriamente
unlabeled_indices = np.random.rand(len(y_train_semi)) < unlabeled_percentage

# Remove os rótulos
y_train_semi[unlabeled_indices] = -1

print(f"Total de amostras: {len(y_train_semi)}")
print(f"Amostras rotuladas: {(y_train_semi != -1).sum()}")
print(f"Amostras sem rótulo: {(y_train_semi == -1).sum()}")

Total de amostras: 5634
Amostras rotuladas: 1651
Amostras sem rótulo: 3983


## Self Training (Pseudo Labeling)

O Self Training é uma abordagem semi-supervisionada em que um modelo é inicialmente treinado apenas com os dados rotulados.

Em seguida, esse modelo realiza previsões sobre os dados sem rótulo.

As previsões com maior confiança passam a ser utilizadas como novos rótulos (pseudo-rótulos), permitindo um novo treinamento utilizando uma quantidade maior de exemplos.}

In [62]:
from sklearn.base import clone

xgb_base = clone(model_xgb)

In [63]:
X_train_transformed = preprocessor.fit_transform(X_train)
X_test_transformed = preprocessor.transform(X_test)

print("Treino:", X_train_transformed.shape)
print("Teste :", X_test_transformed.shape)

Treino: (5634, 5320)
Teste : (1409, 5320)


In [64]:
from xgboost import XGBClassifier

xgb_semi = XGBClassifier(
    n_estimators=200,
    learning_rate=0.1,
    max_depth=4,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    eval_metric="logloss"
)

In [65]:
self_training_model = SelfTrainingClassifier(
    estimator=xgb_semi,
    threshold=0.90,
    verbose=True
)

self_training_model.fit(
    X_train_transformed,
    y_train_semi
)

End of iteration 1, added 1769 new labels.
End of iteration 2, added 358 new labels.
End of iteration 3, added 164 new labels.
End of iteration 4, added 110 new labels.
End of iteration 5, added 99 new labels.
End of iteration 6, added 51 new labels.
End of iteration 7, added 37 new labels.
End of iteration 8, added 35 new labels.
End of iteration 9, added 41 new labels.
End of iteration 10, added 21 new labels.


,"estimator estimator: estimator objectAn estimator object implementing `fit` and `predict_proba`.Invoking the `fit` method will fit a clone of the passed estimator,which will be stored in the `estimator_` attribute... versionadded:: 1.6 `estimator` was added to replace `base_estimator`.","XGBClassifier...ree=None, ...)"
,"threshold threshold: float, default=0.75The decision threshold for use with `criterion='threshold'`.Should be in [0, 1). When using the `'threshold'` criterion, a:ref:`well calibrated classifier <calibration>` should be used.",0.9
,"verbose verbose: bool, default=FalseEnable verbose output.",True
,"criterion criterion: {'threshold', 'k_best'}, default='threshold'The selection criterion used to select which labels to add to thetraining set. If `'threshold'`, pseudo-labels with predictionprobabilities above `threshold` are added to the dataset. If `'k_best'`,the `k_best` pseudo-labels with highest prediction probabilities areadded to the dataset. When using the 'threshold' criterion, a:ref:`well calibrated classifier <calibration>` should be used.",'threshold'
,"k_best k_best: int, default=10The amount of samples to add in each iteration. Only used when`criterion='k_best'`.",10
,"max_iter max_iter: int or None, default=10Maximum number of iterations allowed. Should be greater than or equalto 0. If it is `None`, the classifier will continue to predict labelsuntil no new pseudo-labels are added, or all unlabeled samples havebeen labeled.",10
Name,Type,Value
"classes_ classes_: ndarray or list of ndarray of shape (n_classes,)Class labels for each output. (Taken from the trained`estimator_`).","ndarray[int64](2,)","[0,1]"
estimator_ estimator_: estimator objectThe fitted estimator.,XGBClassifier,"XGBClassifier...ree=None, ...)"
"labeled_iter_ labeled_iter_: ndarray of shape (n_samples,)The iteration in which each sample was labeled. When a sample hasiteration 0, the sample was already labeled in the original dataset.When a sample has iteration -1, the sample was not labeled in anyiteration.","ndarray[int64](5634,)","[-1, 0, 0,...,-1, 1, 1]"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`... versionadded:: 0.24,int,5320


In [66]:
y_pred_self = self_training_model.predict(X_test_transformed)

In [67]:
acc_self = accuracy_score(y_test, y_pred_self)

print(f"Acurácia - Self Training: {acc_self:.4f}")

print("\nClassification Report\n")
print(classification_report(y_test, y_pred_self))

Acurácia - Self Training: 0.7793

Classification Report

              precision    recall  f1-score   support

           0       0.84      0.87      0.85      1035
           1       0.59      0.54      0.56       374

    accuracy                           0.78      1409
   macro avg       0.72      0.70      0.71      1409
weighted avg       0.77      0.78      0.78      1409



## Resultados do Self Training

O algoritmo Self Training foi aplicado utilizando o XGBoost como estimador base e uma simulação de dados não rotulados.

Os resultados serão comparados com os modelos supervisionados desenvolvidos anteriormente para avaliar se o uso de pseudo-rótulos contribuiu para a melhoria do desempenho do modelo.